In [1]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import cupy as cp
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from cuml.ensemble import RandomForestClassifier as cuRF, RandomForestRegressor as cuRFR
from cuml.linear_model import LogisticRegression as cuLogisticRegression, LinearRegression as cuLinearRegression
from cuml.svm import SVC as cuSVC, SVR as cuSVR
from cuml.model_selection import train_test_split as cu_train_test_split
from cuml.metrics import accuracy_score as cu_accuracy_score
import warnings
warnings.filterwarnings("ignore", message="Random state is currently ignored by probabilistic SVC")

try:
    from cuml.ensemble import GradientBoostingClassifier as cuGBC, GradientBoostingRegressor as cuGBR
    HAS_GRADIENT_BOOSTING = True
except ImportError:
    print("cuML Gradient Boosting 不可用，将跳过该模型")
    HAS_GRADIENT_BOOSTING = False

import xgboost as xgb
from datetime import datetime
from pandas.api.types import CategoricalDtype
import traceback
from sklearn.base import clone, BaseEstimator
from sklearn.model_selection import KFold
from typing import Dict, Any, Tuple, Optional, List
import warnings
import json
warnings.filterwarnings('ignore')

from docx import Document
from docx.shared import Inches

import copy

os.environ['XGBOOST_DEVICE_MEMORY_LIMIT'] = '10GB'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# ========================
# GPU资源清理增强模块
# ========================
def clear_gpu_resources():
    gc.collect()
    time.sleep(0.5)
    
    if 'cp' in globals():
        try:
            cp._default_memory_pool.free_all_blocks()
            cp.get_default_memory_pool().free_all_blocks()
        except Exception: 
            pass
        
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

def monitor_gpu_memory(threshold=0.8):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated()
        total = torch.cuda.get_device_properties(0).total_memory
        ratio = allocated / total
        if ratio > threshold:
            print(f"GPU内存使用率超过{threshold*100:.1f}% ({allocated/1e9:.2f}GB/{total/1e9:.2f}GB)，执行清理...")
            clear_gpu_resources()
            return True
    return False

# ========================
# 第一阶段模型工厂类
# ========================
class FirstStageModelFactory:
    def __init__(self):
        self.model_configs = {
            'RandomForest': {
                'classification': lambda: cuRF(n_estimators=200, max_depth=20, random_state=42),
                'regression': lambda: cuRFR(n_estimators=200, max_depth=20, random_state=42)
            },
            'LogisticRegression': {
                'classification': lambda: cuLogisticRegression(penalty='l2', C=1.0, max_iter=1000),
                'regression': lambda: cuLinearRegression(fit_intercept=True, algorithm='eig')
            },
            'GradientBoosting': {
                'classification': lambda: cuGBC(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None,
                'regression': lambda: cuGBR(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None
            },
            'SVC': {
                'classification': lambda: cuSVC(kernel='rbf', C=1.0, probability=True, random_state=42),
                'regression': lambda: cuSVR(kernel='rbf', C=1.0)
            },
            'XGBoost': {
                'classification': lambda: xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist'),
                'regression': lambda: xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist')
            },
            'MLP': {
                'classification': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='classification'),
                'regression': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='regression')
            }
        }
    
    def get_model(self, model_name, task_type, input_size=None):
        if model_name not in self.model_configs:
            raise ValueError(f"不支持的模型类型: {model_name}")
        
        if task_type not in self.model_configs[model_name]:
            raise ValueError(f"模型 {model_name} 不支持任务类型: {task_type}")
        
        if model_name == 'MLP':
            if input_size is None:
                raise ValueError("MLP模型需要指定input_size参数")
            return self.model_configs[model_name][task_type](input_size)
        else:
            model = self.model_configs[model_name][task_type]()
            if model is None:
                raise ValueError(f"模型 {model_name} 不可用，可能缺少依赖")
            return model

# ========================
# PyTorch MLP模型
# ========================
class PyTorchMLP(nn.Module):
    def __init__(self, input_size, hidden_sizes=[100, 50], output_size=1, 
                 task_type='classification', dropout_rate=0.1):
        super(PyTorchMLP, self).__init__()
        self.task_type = task_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        
        if task_type == 'classification':
            layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers).to(self.device)
        
    def forward(self, x):
        return self.network(x)
    
    def fit(self, X, y, epochs=100, batch_size=256, lr=0.001):
        self.train()
        
        if not isinstance(X, torch.Tensor):
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        else:
            X_tensor = X.to(self.device)
            
        if not isinstance(y, torch.Tensor):
            y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device)
        else:
            y_tensor = y.to(self.device)
        
        if self.task_type == 'classification':
            y_tensor = y_tensor.view(-1, 1)
        
        if self.task_type == 'classification':
            criterion = nn.BCELoss()
        else:
            criterion = nn.MSELoss()
        
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch_X, batch_y in dataloader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if (epoch + 1) % 20 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")
    
    def predict(self, X):
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            predictions = self(X_tensor)
            
            if self.task_type == 'classification':
                return (predictions > 0.5).float().cpu().numpy().flatten()
            else:
                return predictions.cpu().numpy().flatten()
    
    def predict_proba(self, X):
        if self.task_type != 'classification':
            raise ValueError("predict_proba仅适用于分类任务")
        
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            return self(X_tensor).cpu().numpy()

# ========================
# 使用交叉验证计算残差的辅助函数
# ========================
def get_residuals(model_proto, X, target, task_type, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(target))
    
    for train_idx, test_idx in kf.split(X):
        if isinstance(model_proto, PyTorchMLP):
            model_clone = copy.deepcopy(model_proto)
        else:
            model_clone = clone(model_proto)
        
        X_train, X_test = X[train_idx], X[test_idx]
        target_train, target_test = target[train_idx], target[test_idx]
        
        model_clone.fit(X_train, target_train)
        
        if task_type == 'classification':
            if hasattr(model_clone, 'predict_proba'):
                pred = model_clone.predict_proba(X_test)[:, 1] if model_clone.predict_proba(X_test).ndim > 1 else model_clone.predict_proba(X_test)
            else:
                pred = model_clone.predict(X_test)
        else:
            pred = model_clone.predict(X_test)
        
        residuals[test_idx] = target_test - pred
    
    return residuals

# ========================
# 第二阶段IV分析（简单2SLS on residuals）
# ========================
from scipy import stats
def second_stage_iv_analysis(y_res, d_res, z_res, n_bootstrap=1000):
    n = len(y_res)
    
    # 第一阶段：D_res ~ Z_res (no constant)
    z_res = z_res.reshape(-1, 1)
    d_res = d_res.reshape(-1, 1)
    y_res = y_res.reshape(-1, 1)
    
    # 第一阶段系数
    alpha = np.dot(z_res.T, d_res) / np.dot(z_res.T, z_res)
    d_hat = alpha * z_res
    
    # 第一阶段F统计量 (for weak IV test)
    ss_res_first = np.sum((d_res - d_hat)**2)
    ss_tot_first = np.sum(d_res**2)  # since mean zero
    r2_first = 1 - ss_res_first / ss_tot_first if ss_tot_first > 0 else 0
    f_stat_first = (r2_first / (1 - r2_first)) * (n - 1) if (1 - r2_first) > 0 else 0
    f_p_first = 1 - stats.f.cdf(f_stat_first, 1, n-1) if n > 1 else 1.0
    
    # 缩减形式：Y_res ~ Z_res
    beta_red = np.dot(z_res.T, y_res) / np.dot(z_res.T, z_res)
    y_hat_red = beta_red * z_res
    ss_res_red = np.sum((y_res - y_hat_red)**2)
    ss_tot_red = np.sum(y_res**2)
    r2_red = 1 - ss_res_red / ss_tot_red if ss_tot_red > 0 else 0
    f_stat_red = (r2_red / (1 - r2_red)) * (n - 1) if (1 - r2_red) > 0 else 0
    f_p_red = 1 - stats.f.cdf(f_stat_red, 1, n-1) if n > 1 else 1.0
    
    # 第二阶段：Y_res ~ D_hat (no constant)
    theta = np.dot(d_hat.T, y_res) / np.dot(d_hat.T, d_hat)
    effect = theta.item()
    
    # Bootstrap for SE and CI
    bootstrap_effects = []
    for _ in range(n_bootstrap):
        indices = np.random.choice(n, n, replace=True)
        z_boot = z_res[indices]
        d_boot = d_res[indices]
        y_boot = y_res[indices]
        
        alpha_boot = np.dot(z_boot.T, d_boot) / np.dot(z_boot.T, z_boot)
        d_hat_boot = alpha_boot * z_boot
        
        theta_boot = np.dot(d_hat_boot.T, y_boot) / np.dot(d_hat_boot.T, d_hat_boot)
        bootstrap_effects.append(theta_boot.item())
    
    se = np.std(bootstrap_effects, ddof=1)
    ci_lower = np.percentile(bootstrap_effects, 2.5)
    ci_upper = np.percentile(bootstrap_effects, 97.5)
    
    # t-stat and p-value
    t_stat = effect / se if se > 0 else 0
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), n-1))
    
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_stat,
        'p_value': p_value,
        'first_stage_alpha': alpha.item(),
        'first_stage_f_stat': f_stat_first,
        'first_stage_f_p': f_p_first,
        'reduced_beta': beta_red.item(),
        'reduced_f_stat': f_stat_red,
        'reduced_f_p': f_p_red
    }

# ========================
# 第一阶段训练函数
# ========================
def first_stage_cuml_enhanced(X, target, task_type='regression', model_name='RandomForest', test_size=0.25):
    clear_gpu_resources()
    
    X_train, X_test, target_train, target_test = cu_train_test_split(
        X, target, test_size=test_size, random_state=42
    )
    
    model_factory = FirstStageModelFactory()
    
    try:
        if model_name == 'MLP':
            input_size = X_train.shape[1]
            model = model_factory.get_model(model_name, task_type, input_size)
        else:
            model = model_factory.get_model(model_name, task_type)
        
        if model_name == 'MLP':
            X_train_pt = torch.tensor(cp.asnumpy(X_train), dtype=torch.float32)
            target_train_pt = torch.tensor(cp.asnumpy(target_train), dtype=torch.float32)
            model.fit(X_train_pt, target_train_pt)
            X_test_pt = torch.tensor(cp.asnumpy(X_test), dtype=torch.float32)
            pred = model.predict(X_test_pt)
            pred = cp.array(pred)
        else:
            model.fit(X_train, target_train)
            pred = model.predict(X_test)
        
        metrics = evaluate_model(model, X_test, target_test, model_name, task_type)
        
        if task_type == 'classification':
            summary = f"准确率: {metrics.get('accuracy', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
        else:
            summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
        
        print(f"{model_name} {task_type}模型评估 - {summary}")
        
        return model, metrics
        
    except Exception as e:
        print(f"训练{model_name}模型失败: {str(e)}")
        traceback.print_exc()
        return None, None

# ========================
# 评估指标计算函数
# ========================
def evaluate_model(model, X_test, y_test, model_name="Model", task_type='classification'):
    try:
        if hasattr(X_test, 'get'):
            X_test_np = X_test.get()
        elif hasattr(X_test, 'cpu'):
            X_test_np = X_test.cpu().numpy()
        elif hasattr(X_test, 'numpy'):
            X_test_np = X_test.numpy()
        else:
            X_test_np = np.array(X_test)
            
        if hasattr(y_test, 'get'):
            y_test_np = y_test.get()
        elif hasattr(y_test, 'cpu'):
            y_test_np = y_test.cpu().numpy()
        elif hasattr(y_test, 'numpy'):
            y_test_np = y_test.numpy()
        else:
            y_test_np = np.array(y_test)
        
        y_pred = model.predict(X_test_np)
        
        if hasattr(y_pred, 'get'):
            y_pred_np = y_pred.get()
        elif hasattr(y_pred, 'cpu'):
            y_pred_np = y_pred.cpu().numpy()
        elif hasattr(y_pred, 'numpy'):
            y_pred_np = y_pred.numpy()
        else:
            y_pred_np = np.array(y_pred)
        
        metrics = {}
        
        if task_type == 'regression':
            metrics['r2'] = r2_score(y_test_np, y_pred_np)
            metrics['mse'] = mean_squared_error(y_test_np, y_pred_np)
            metrics['rmse'] = np.sqrt(metrics['mse'])
            metrics['mae'] = mean_absolute_error(y_test_np, y_pred_np)
            
            print(f"{model_name} 回归模型评估 - R²: {metrics['r2']:.4f}, RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}")
            
        else:
            metrics['accuracy'] = accuracy_score(y_test_np, y_pred_np)
            metrics['precision'] = precision_score(y_test_np, y_pred_np, average='weighted')
            metrics['recall'] = recall_score(y_test_np, y_pred_np, average='weighted')
            metrics['f1'] = f1_score(y_test_np, y_pred_np, average='weighted')
            
            auc_score = np.nan
            try:
                if hasattr(model, 'predict_proba'):
                    y_proba = model.predict_proba(X_test_np)
                    if hasattr(y_proba, 'get'):
                        y_proba = y_proba.get()
                    elif hasattr(y_proba, 'cpu'):
                        y_proba = y_proba.cpu().numpy()
                    
                    if len(y_proba.shape) > 1 and y_proba.shape[1] == 2:
                        auc_score = roc_auc_score(y_test_np, y_proba[:, 1])
                    elif len(y_proba.shape) > 1:
                        auc_score = roc_auc_score(y_test_np, y_proba, multi_class='ovr')
                    else:
                        auc_score = roc_auc_score(y_test_np, y_proba)
                elif hasattr(model, 'decision_function'):
                    y_score = model.decision_function(X_test_np)
                    if hasattr(y_score, 'get'):
                        y_score = y_score.get()
                    elif hasattr(y_score, 'cpu'):
                        y_score = y_score.cpu().numpy()
                    auc_score = roc_auc_score(y_test_np, y_score)
                else:
                    auc_score = np.nan
            except Exception:
                auc_score = np.nan
            
            metrics['auc'] = auc_score
            print(f"{model_name} 分类模型评估 - 准确率: {metrics['accuracy']:.4f}, 精确率: {metrics['precision']:.4f}, 召回率: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, AUC: {metrics['auc']:.4f}")
        
        metrics['predictions'] = y_pred_np
        return metrics
        
    except Exception as e:
        print(f"评估模型 {model_name} 时出错: {str(e)}")
        traceback.print_exc()
        if task_type == 'regression':
            return {'r2': np.nan, 'mse': np.nan, 'rmse': np.nan, 'mae': np.nan, 'predictions': None}
        else:
            return {'accuracy': np.nan, 'precision': np.nan, 'recall': np.nan, 'f1': np.nan, 'auc': np.nan, 'predictions': None}

# ========================
# 模型比较器类
# ========================
class ModelComparator:
    def __init__(self, task_type):
        self.task_type = task_type
        self.model_metrics = {}
    
    def add_model(self, model_name, metrics):
        self.model_metrics[model_name] = metrics
    
    def _calculate_score(self, metrics):
        if self.task_type == 'regression':
            r2 = metrics.get('r2', 0)
            rmse = metrics.get('rmse', 1e6)
            mae = metrics.get('mae', 1e6)
            if r2 < 0:
                return -1.0 * abs(r2)
            score = r2 * 0.7 + (1 / (rmse + 1e-6)) * 0.15 + (1 / (mae + 1e-6)) * 0.15
            return score
        else:
            accuracy = metrics.get('accuracy', 0)
            precision = metrics.get('precision', 0)
            recall = metrics.get('recall', 0)
            f1 = metrics.get('f1', 0)
            auc = metrics.get('auc', 0)
            if np.isnan(auc):
                auc = 0
            if auc > 0:
                score = accuracy * 0.2 + precision * 0.2 + recall * 0.2 + f1 * 0.2 + auc * 0.2
            else:
                score = accuracy * 0.25 + precision * 0.25 + recall * 0.25 + f1 * 0.25
            return score
    
    def compare_models(self):
        model_scores = {}
        for model_name, metrics in self.model_metrics.items():
            score = self._calculate_score(metrics)
            model_scores[model_name] = score
        sorted_models = sorted(model_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_models
    
    def get_best_model(self):
        sorted_models = self.compare_models()
        return sorted_models[0][0] if sorted_models else None
    
    def print_comparison(self):
        print(f"\n=== {self.task_type.upper()} 模型比较结果 ===")
        sorted_models = self.compare_models()
        for rank, (model_name, score) in enumerate(sorted_models, 1):
            metrics = self.model_metrics[model_name]
            if self.task_type == 'classification':
                summary = f"准确率: {metrics.get('accuracy', 0):.4f}, 精确率: {metrics.get('precision', 0):.4f}, 召回率: {metrics.get('recall', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
            else:
                summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
            print(f"{rank}. {model_name}: 得分={score:.4f}, {summary}")

# ========================
# 数据预处理封装
# ========================
class DataPreprocessor:
    def __init__(self, control_variables):
        self.control_variables = control_variables
        self.scalers = {}
        self.imputers = {}
        self.encoders = {}
    
    def load_and_preprocess_data(self, file_path, y_var, d_var, z_var):
        print("步骤1: 数据准备")
        print(f"使用文件: {file_path}")
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件 '{file_path}' 不存在")
        
        data = pd.read_stata(file_path)
        
        y_series = data[y_var]
        d_series = data[d_var]
        z_series = data[z_var]
        
        le = LabelEncoder()
        y = le.fit_transform(y_series) if y_series.dtype == 'object' or isinstance(y_series.dtype, CategoricalDtype) else y_series.values
        d = le.fit_transform(d_series) if d_series.dtype == 'object' or isinstance(d_series.dtype, CategoricalDtype) else d_series.values
        z = z_series.values  # continuous
        
        missing_cols = [col for col in self.control_variables if col not in data.columns]
        if missing_cols:
            raise ValueError(f"数据中缺少以下控制变量: {missing_cols}")
        
        for col in self.control_variables:
            if data[col].dtype == 'object' or isinstance(data[col].dtype, CategoricalDtype):
                self.encoders[col] = LabelEncoder()
                data[col] = self.encoders[col].fit_transform(data[col].astype(str))
            
            if data[col].isnull().sum() > 0:
                self.imputers[col] = SimpleImputer(strategy='mean')
                data[col] = self.imputers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()
            
            self.scalers[col] = StandardScaler()
            data[col] = self.scalers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()

        # Preprocess Z if needed
        if data[z_var].isnull().sum() > 0:
            z_imputer = SimpleImputer(strategy='mean')
            z = z_imputer.fit_transform(z.reshape(-1, 1)).flatten()
        z_scaler = StandardScaler()
        z = z_scaler.fit_transform(z.reshape(-1, 1)).flatten()
        
        X = data[self.control_variables].values
        
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        d_gpu = cp.array(d, dtype=cp.float32)
        z_gpu = cp.array(z, dtype=cp.float32)
        
        print("检查数据完整性：")
        print("X 中是否有 NaN 或 Inf:", cp.any(cp.isnan(X_gpu)), cp.any(cp.isinf(X_gpu)))
        print("y 中是否有 NaN 或 Inf:", cp.any(cp.isnan(y_gpu)), cp.any(cp.isinf(y_gpu)))
        print("d 中是否有 NaN 或 Inf:", cp.any(cp.isnan(d_gpu)), cp.any(cp.isinf(d_gpu)))
        print("z 中是否有 NaN 或 Inf:", cp.any(cp.isnan(z_gpu)), cp.any(cp.isinf(z_gpu)))
        
        X_gpu = cp.nan_to_num(X_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        y_gpu = cp.nan_to_num(y_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        d_gpu = cp.nan_to_num(d_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        z_gpu = cp.nan_to_num(z_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        
        print(f"数据形状: X={X_gpu.shape}, y={y_gpu.shape}, d={d_gpu.shape}, z={z_gpu.shape}")
        
        return cp.asnumpy(X_gpu), cp.asnumpy(y_gpu), cp.asnumpy(d_gpu), cp.asnumpy(z_gpu), data

# ========================
# 保存结果到Word
# ========================
def save_iv_results(iv_results, model_y_metrics, model_d_metrics, model_z_metrics, 
                    best_y_model, best_d_model, best_z_model, save_path):
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    doc = Document()
    doc.add_heading('IV-DML 分析结果', 0)
    
    doc.add_heading('主要IV估计', level=1)
    para = doc.add_paragraph()
    para.add_run(f"处理效应 (theta): {iv_results['effect']:.6f}\n")
    para.add_run(f"标准误差: {iv_results['se']:.6f}\n")
    para.add_run(f"95%置信区间: [{iv_results['lb']:.6f}, {iv_results['ub']:.6f}]\n")
    para.add_run(f"t统计量: {iv_results['t_statistic']:.4f}\n")
    para.add_run(f"p值: {iv_results['p_value']:.6f}")
    
    doc.add_heading('第一阶段 (工具变量与自变量关系)', level=1)
    para = doc.add_paragraph()
    para.add_run(f"系数 (alpha): {iv_results['first_stage_alpha']:.6f}\n")
    para.add_run(f"F统计量: {iv_results['first_stage_f_stat']:.4f}\n")
    para.add_run(f"F p值: {iv_results['first_stage_f_p']:.6f}")
    
    doc.add_heading('缩减形式 (工具变量与因变量关系)', level=1)
    para = doc.add_paragraph()
    para.add_run(f"系数 (beta): {iv_results['reduced_beta']:.6f}\n")
    para.add_run(f"F统计量: {iv_results['reduced_f_stat']:.4f}\n")
    para.add_run(f"F p值: {iv_results['reduced_f_p']:.6f}")
    
    doc.add_heading('模型信息', level=1)
    para = doc.add_paragraph()
    para.add_run(f"最佳Y模型: {best_y_model}\n")
    para.add_run(f"最佳D模型: {best_d_model}\n")
    para.add_run(f"最佳Z模型: {best_z_model}")
    
    doc.add_heading('Y模型指标 (分类)', level=2)
    para = doc.add_paragraph()
    para.add_run(f"准确率: {model_y_metrics.get('accuracy', np.nan):.4f}\n")
    para.add_run(f"精确率: {model_y_metrics.get('precision', np.nan):.4f}\n")
    para.add_run(f"召回率: {model_y_metrics.get('recall', np.nan):.4f}\n")
    para.add_run(f"F1: {model_y_metrics.get('f1', np.nan):.4f}\n")
    para.add_run(f"AUC: {model_y_metrics.get('auc', np.nan):.4f}")
    
    doc.add_heading('D模型指标 (分类)', level=2)
    para = doc.add_paragraph()
    para.add_run(f"准确率: {model_d_metrics.get('accuracy', np.nan):.4f}\n")
    para.add_run(f"精确率: {model_d_metrics.get('precision', np.nan):.4f}\n")
    para.add_run(f"召回率: {model_d_metrics.get('recall', np.nan):.4f}\n")
    para.add_run(f"F1: {model_d_metrics.get('f1', np.nan):.4f}\n")
    para.add_run(f"AUC: {model_d_metrics.get('auc', np.nan):.4f}")
    
    doc.add_heading('Z模型指标 (回归)', level=2)
    para = doc.add_paragraph()
    para.add_run(f"R²: {model_z_metrics.get('r2', np.nan):.4f}\n")
    para.add_run(f"RMSE: {model_z_metrics.get('rmse', np.nan):.4f}\n")
    para.add_run(f"MAE: {model_z_metrics.get('mae', np.nan):.4f}")
    
    os.makedirs(save_path, exist_ok=True)
    word_file = os.path.join(save_path, f"iv_dml_results_{timestamp_str}.docx")
    doc.save(word_file)
    print(f"结果已保存到Word文件: {word_file}")
    
    return word_file

# ========================
# 主程序
# ========================
if __name__ == "__main__":
    try:
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.8)
            
        print("正在执行IV-DML处理流程")
        
        control_variables = [
            'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
            'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
            'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
            'intelligence', 'person_status', 'party', 'workasso',
            'fid','pid','year',
            'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
            'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
            'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
            'intelligence_sq', 'person_status_sq', 
            'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
            'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
            'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
            'intelligence_cub', 'person_status_cub'
        ]
        
        model_names = ['RandomForest', 'LogisticRegression', 'SVC', 'XGBoost', 'MLP']
        if HAS_GRADIENT_BOOSTING:
            model_names.append('GradientBoosting')
        
        save_path = "D:\\my-rapids-project\\result"
        os.makedirs(save_path, exist_ok=True)
        
        preprocessor = DataPreprocessor(control_variables)
        X, y, d, z, original_data = preprocessor.load_and_preprocess_data("l3dml_data_digital.dta", 'entrepreneurship', 'mobile_use', 'ln_durables_asset')
        
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        d_gpu = cp.array(d, dtype=cp.float32)
        z_gpu = cp.array(z, dtype=cp.float32)
        
        y_comparator = ModelComparator('classification')
        d_comparator = ModelComparator('classification')
        z_comparator = ModelComparator('regression')
        
        print("\n=== 训练并比较Y模型（分类任务）===")
        y_models = {}
        y_metrics_dict = {}
        for model_name in model_names:
            print(f"\n训练Y模型: {model_name}")
            model, metrics = first_stage_cuml_enhanced(X_gpu, y_gpu, task_type='classification', model_name=model_name)
            if model is not None and metrics is not None:
                y_models[model_name] = model
                y_metrics_dict[model_name] = metrics
                y_comparator.add_model(model_name, metrics)
        y_comparator.print_comparison()
        
        print("\n=== 训练并比较D模型（分类任务）===")
        d_models = {}
        d_metrics_dict = {}
        for model_name in model_names:
            print(f"\n训练D模型: {model_name}")
            model, metrics = first_stage_cuml_enhanced(X_gpu, d_gpu, task_type='classification', model_name=model_name)
            if model is not None and metrics is not None:
                d_models[model_name] = model
                d_metrics_dict[model_name] = metrics
                d_comparator.add_model(model_name, metrics)
        d_comparator.print_comparison()
        
        print("\n=== 训练并比较Z模型（回归任务）===")
        z_models = {}
        z_metrics_dict = {}
        for model_name in model_names:
            print(f"\n训练Z模型: {model_name}")
            model, metrics = first_stage_cuml_enhanced(X_gpu, z_gpu, task_type='regression', model_name=model_name)
            if model is not None and metrics is not None:
                z_models[model_name] = model
                z_metrics_dict[model_name] = metrics
                z_comparator.add_model(model_name, metrics)
        z_comparator.print_comparison()
        
        best_y_model_name = y_comparator.get_best_model()
        best_d_model_name = d_comparator.get_best_model()
        best_z_model_name = z_comparator.get_best_model()
        
        if best_y_model_name is None or best_d_model_name is None or best_z_model_name is None:
            print("未能成功训练任何模型，程序退出")
            exit(1)
        
        print(f"\n选择的最佳Y模型: {best_y_model_name}")
        print(f"选择的最佳D模型: {best_d_model_name}")
        print(f"选择的最佳Z模型: {best_z_model_name}")
        
        best_model_y = y_models[best_y_model_name]
        best_model_d = d_models[best_d_model_name]
        best_model_z = z_models[best_z_model_name]
        
        best_metrics_y = y_comparator.model_metrics[best_y_model_name]
        best_metrics_d = d_comparator.model_metrics[best_d_model_name]
        best_metrics_z = z_comparator.model_metrics[best_z_model_name]
        
        print("\n执行交叉拟合计算残差 (5折)...")
        y_res = get_residuals(best_model_y, X, y, 'classification', n_splits=5)
        d_res = get_residuals(best_model_d, X, d, 'classification', n_splits=5)
        z_res = get_residuals(best_model_z, X, z, 'regression', n_splits=5)
        
        print("\n执行第二阶段IV分析...")
        iv_results = second_stage_iv_analysis(y_res, d_res, z_res, n_bootstrap=1000)
        
        print("\nIV-DML结果:")
        print(f"处理效应: {iv_results['effect']:.6f}")
        print(f"标准误差: {iv_results['se']:.6f}")
        print(f"95% CI: [{iv_results['lb']:.6f}, {iv_results['ub']:.6f}]")
        print(f"t统计量: {iv_results['t_statistic']:.4f}")
        print(f"p值: {iv_results['p_value']:.6f}")
        
        print("\n第一阶段 (工具变量与自变量关系):")
        print(f"系数: {iv_results['first_stage_alpha']:.6f}")
        print(f"F统计量: {iv_results['first_stage_f_stat']:.4f}")
        print(f"F p值: {iv_results['first_stage_f_p']:.6f}")
        
        print("\n缩减形式 (工具变量与因变量关系):")
        print(f"系数: {iv_results['reduced_beta']:.6f}")
        print(f"F统计量: {iv_results['reduced_f_stat']:.4f}")
        print(f"F p值: {iv_results['reduced_f_p']:.6f}")
        
        save_iv_results(iv_results, best_metrics_y, best_metrics_d, best_metrics_z, 
                        best_y_model_name, best_d_model_name, best_z_model_name, save_path)
        
        print("分析完成！")
        
    except Exception as e:
        print(f"程序执行出错: {str(e)}")
        traceback.print_exc()
        
    finally:
        print("执行资源清理...")
        clear_gpu_resources()
        print("程序退出！")

cuML Gradient Boosting 不可用，将跳过该模型
正在执行IV-DML处理流程
步骤1: 数据准备
使用文件: l3dml_data_digital.dta
检查数据完整性：
X 中是否有 NaN 或 Inf: False False
y 中是否有 NaN 或 Inf: False False
d 中是否有 NaN 或 Inf: False False
z 中是否有 NaN 或 Inf: False False
数据形状: X=(26354, 53), y=(26354,), d=(26354,), z=(26354,)

=== 训练并比较Y模型（分类任务）===

训练Y模型: RandomForest
RandomForest 分类模型评估 - 准确率: 0.9651, 精确率: 0.9553, 召回率: 0.9651, F1: 0.9508, AUC: 0.8595
RandomForest classification模型评估 - 准确率: 0.9651, F1: 0.9508, AUC: 0.8595

训练Y模型: LogisticRegression
LogisticRegression 分类模型评估 - 准确率: 0.9645, 精确率: 0.9516, 召回率: 0.9645, F1: 0.9499, AUC: 0.8503
LogisticRegression classification模型评估 - 准确率: 0.9645, F1: 0.9499, AUC: 0.8503

训练Y模型: SVC
[2025-10-25 09:47:40.218] [CUML] [warning] Random state is currently ignored by probabilistic SVC
SVC 分类模型评估 - 准确率: 0.9630, 精确率: 0.9429, 召回率: 0.9630, F1: 0.9480, AUC: 0.7633
SVC classification模型评估 - 准确率: 0.9630, F1: 0.9480, AUC: 0.7633

训练Y模型: XGBoost
XGBoost 分类模型评估 - 准确率: 0.9658, 精确率: 0.9573, 召回率: 0.9658, F1: 0.9591, 

In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels.iv import IV2SLS  # 如果有 linearmodels 库，否则用 statsmodels
from scipy import stats

# 假设数据文件路径
file_path = "l3dml_data_digital.dta"

# 控制变量列表（从原代码复制）
control_variables = [
    'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
    'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
    'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
    'intelligence', 'person_status', 'party', 'workasso',
    'fid','pid','year',
    'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
    'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
    'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
    'intelligence_sq', 'person_status_sq', 
    'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
    'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
    'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
    'intelligence_cub', 'person_status_cub'
]

# 变量定义
y_var = 'entrepreneurship'
d_var = 'mobile_use'
z_var = 'ln_durables_asset'

# 加载数据，禁用分类转换以获取原始数值
data = pd.read_stata(file_path, convert_categoricals=False)

# 简单预处理（假设 y, d 为数值型，可能包含缺失值编码）
# 处理缺失值
data = data.dropna(subset=[y_var, d_var, z_var] + control_variables)

# 确保 y, d 是整数（假设原始为 0/1 或类似）
data[y_var] = data[y_var].astype(int)
data[d_var] = data[d_var].astype(int)

# 添加常数项
data['const'] = 1.0

# 定义变量
Y = data[y_var]
D = data[d_var]
Z = data[z_var]
X = data[control_variables + ['const']]  # 包括控制变量和常数

# 第一阶段：D ~ Z + X（由于 X 已包含 const，不再添加）
first_exog = pd.concat([Z, X], axis=1)
model_first = sm.OLS(D, first_exog).fit()
D_hat = model_first.fittedvalues

# 计算第一阶段 F 统计量（针对工具变量 Z）
# 限制模型：D ~ X (无 Z)
model_restricted = sm.OLS(D, X).fit()
f_stat = ((model_restricted.ssr - model_first.ssr) / 1) / (model_first.ssr / model_first.df_resid)
f_p = 1 - stats.f.cdf(f_stat, 1, model_first.df_resid)

print(f"第一阶段 F 统计量: {f_stat:.4f}, p 值: {f_p:.6f}")

# Stock-Yogo 临界值检查（手动阈值，>16.38 为强（5% 临界 for 10% bias），>10 中等）
if f_stat > 16.38:
    print("工具变量强 (Stock-Yogo 5% 临界值 for 10% bias)")
elif f_stat > 10:
    print("工具变量中等强度")
else:
    print("潜在弱工具变量风险")

# 第二阶段：Y ~ D_hat + X（无额外 const）
second_exog = pd.concat([D_hat, X], axis=1)
model_second = sm.OLS(Y, second_exog).fit()
print("\n第二阶段结果:")
print(model_second.summary())

# Anderson-Rubin 测试（弱 IV 鲁棒置信区间）
# 手动实现简单 AR 测试（假设单 IV）
# H0: beta = 0
ar_stat = (Y - 0 * D).dot(Z) / np.sqrt((Y - 0 * D).var() * Z.var() * len(Y))  # 简化版
ar_p = 2 * (1 - stats.norm.cdf(abs(ar_stat)))
print(f"\nAnderson-Rubin 测试统计量 (for beta=0): {ar_stat:.4f}, p 值: {ar_p:.6f}")

# 如果使用 linearmodels (如果环境有)
try:
    iv_model = IV2SLS(dependent=Y, exog=X.drop(columns=['const'] if 'const' in X.columns else []), endog=D, instruments=Z).fit(cov_type='robust')  # 添加 robust 协方差
    print("\nIV2SLS 结果 (使用 linearmodels):")
    print(iv_model.summary)
    # 弱 IV 诊断 - 修正键为 'f_statistic'
    if hasattr(iv_model.first_stage, 'diagnostics') and 'f_statistic' in iv_model.first_stage.diagnostics:
        print(f"弱 IV F 统计量: {iv_model.first_stage.diagnostics['f_statistic']:.4f}")
    else:
        print("弱 IV F 统计量不可用 - 检查 first_stage.diagnostics 键")
except ImportError:
    print("linearmodels 未安装，跳过高级 IV 模型")
except Exception as e:
    print(f"IV2SLS 执行错误: {str(e)}")

第一阶段 F 统计量: 35.0261, p 值: 0.000000
工具变量强 (Stock-Yogo 5% 临界值 for 10% bias)

第二阶段结果:
                            OLS Regression Results                            
Dep. Variable:       entrepreneurship   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.078
Method:                 Least Squares   F-statistic:                     42.14
Date:                Tue, 07 Oct 2025   Prob (F-statistic):               0.00
Time:                        03:30:06   Log-Likelihood:                 8279.7
No. Observations:               26354   AIC:                        -1.645e+04
Df Residuals:                   26299   BIC:                        -1.600e+04
Df Model:                          54                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------